# Train + evaluate on Colab (T4 GPU). Replace the <PLACEHOLDERS>.

In [ ]:
!git clone <YOUR_REPO_URL> repo && cd repo && pip -q install -r requirements.txt
%cd repo
!bash scripts/download_data.sh data            # LOL + DICM + LIME + ExDark subset, builds data/splits.json
!python scripts/verify_data.py --data data      # checks files + split match configs/splits_reference.json

In [ ]:
# Stage 1: supervised fine-tune on the leakage-safe train split, model selection on the val split
!python train.py --data_root data/LOL --split_json data/splits.json --epochs 100 --bs 8 --out weights/zerodce_rgb.pt
!python train.py --data_root data/LOL --split_json data/splits.json --epochs 100 --bs 8 --hybrid --out weights/zerodce_hybrid.pt
# watch val_psnr in the log; if it is low try --nonref_scale 0.03 or --l1_w 20 (tune on VAL, never on test)

In [ ]:
# Stage 2: final single evaluation on the untouched test split (eval15) + unseen sets
!python evaluate.py --low_dir data/LOL/eval15/low --high_dir data/LOL/eval15/high --ablation --max_side 0 --out results/final/lol_eval15
!python evaluate.py --low_dir data/unseen/DICM --out results/final/unseen_DICM
!python evaluate.py --low_dir data/unseen/LIME --out results/final/unseen_LIME
!python scripts/make_eval_md.py --lol results/final/lol_eval15 --unseen results/final/unseen_DICM results/final/unseen_LIME --usns <USN1,USN2,...> --team <T01> --out eval_P1_<T01>.md